<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026-transformers/blob/main/07_question-answering_v2-kikim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
!pip install "farm-haystack[metrics,inference,elasticsearch]" --no-cache-dir

In [ ]:
!pip uninstall gcsfs
!pip install gcsfs==2024.2.0
!pip install fsspec==2024.2.0
!pip install datasets==2.18.0
!pip install rapidfuzz

In [ ]:
# 코랩을 사용하지 않으면 다음 코드를 주석 처리하세요.
!git clone https://github.com/kikim6114/nlp2026-transformers.git
%cd nlp2026-transformers
from install import *
install_requirements(chapter=7.2)

In [ ]:
%env TOKENIZERS_PARALLELISM=false

In [ ]:
# haystack의 로깅을 끕니다.
import logging
for module in ["farm.utils", "farm.infer", "haystack.reader.farm.FARMReader",
               "farm.modeling.prediction_head", "elasticsearch", "haystack.eval",
               "haystack.document_store.base", "haystack.retriever.base",
               "farm.data_handler.dataset"]:
    module_logger = logging.getLogger(module)
    module_logger.setLevel(logging.ERROR)

# 7. Question Answering

- Extractive QA (이 장에서 중점적으로 다룰 주제)
- Generative(or Abstractive) QA
  - Open Generative QA
  - Closed Generative QA

### QA 시스템의 종류
#### Extractive QA ⟹ 주로 다룰 내용
<img alt="Extractive QA" width="500" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_extractiveQA.png?raw=1" id="Extractive QA"/>

#### Open Generative QA
<img alt="Open Generative QA" width="500" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_OpenGenerativeQA.png?raw=1" id="Open Generative QA"/>

#### Retrieval-Augmented Generation QA
<img alt="Retrieval-Augmented Generation QA" width="500" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_RAG-QA.png?raw=1" id="Retrieval-Augmented Generation QA"/>

#### Closed Generative QA
<img alt="Closed Generative QA" width="500" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_closedGenerativeQA.png?raw=1" id="Closed Generative QA"/>

## 7.1 리뷰 기반 QA 시스템 구축하기

여기서 트랜스포머는 고객 리뷰 텍스트에서 의미를 추출하는 강력한 **독해(reading comprehension) 모델**로 동작하게 된다.

### 7.1.1 데이터셋

#### SubjQA
- 6 분야(여행안내, 음식점, 영화, 책, 전자제품, 식료품)
- 10,000여개 고객 리뷰 기반
- 각 리뷰 별로 질문에 맞는 한 문장 이상의 답변 정보가 포함됨
- 대부분의 질문과 답이 주관적 ⟹ 사실 여부가 명확한 질문의 답을 찾는 것보다 어려움
- 질문의 중요한 부분이 리뷰에 전혀 나타나지 않는다 ⟹ 키워드 검색이나 질문 재구성 등의 방식으로는 답을 찾지 못한다.
- 따라서 SubjQA는 리뷰 기반 QA를 벤치마킹하기에 현실적인 데이터셋이다.

<img alt="Phone with Query" width="400" caption="A question about a product and the corresponding review (the answer span is underlined)" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_phone.png?raw=1" id="phone"/>

In [ ]:
from datasets import get_dataset_config_names

domains = get_dataset_config_names("subjqa", trust_remote_code=True)
domains

In [ ]:
from datasets import load_dataset

subjqa = load_dataset("subjqa", name="electronics")

In [ ]:
subjqa

In [ ]:
subjqa["train"][0]

#### Data Fields
Each domain and split consists of the following columns:

- title: The id of the item/business discussed in the review.
- question: The question (written based on a query opinion).
- id: A unique id assigned to the question-review pair.
- q_reviews_id: A unique id assigned to all question-review pairs with a shared question.
- question_subj_level: The subjectivity level of the question (on a 1 to 5 scale with 1 being the most subjective).
- ques_subj_score: The subjectivity score of the question computed using the TextBlob package.
- context: The review (that mentions the neighboring opinion).
- review_id: A unique id associated with the review.
- answers.text: The span labeled by annotators as the answer.
- answers.answer_start: The (character-level) start index of the answer span highlighted by annotators.
- is_ques_subjective: A boolean subjectivity label derived from question_subj_level (i.e., scores below 4 are considered as subjective)
- answers.answer_subj_level: The subjectivity level of the answer span (on a 1 to 5 scale with 1 being the most subjective).
- answers.ans_subj_score: The subjectivity score of the answer span computed usign the TextBlob package.
- answers.is_ans_subjective: A boolean subjectivity label derived from answer_subj_level (i.e., scores below 4 are considered as subjective)
- domain: The category/domain of the review (e.g., hotels, books, ...).
- nn_mod: The modifier of the neighboring opinion (which appears in the review).
- nn_asp: The aspect of the neighboring opinion (which appears in the review).
- query_mod: The modifier of the query opinion (around which a question is manually written).
- query_asp: The aspect of the query opinion (around which a question is manually written).

In [ ]:
subjqa["train"]["answers"][1]

- `answer_subj_level`: answer span의 주관적인 정도 1 ~ 5 level

- 각 질문에 대한 대답이 Nested disctionary 구조로 저장되어 있다.
- 중첩된 열들을 `flatten()` 메서드를 써서 펼치고 각 분할을 pandas DataFrame으로 변환하자.

In [ ]:
import pandas as pd

dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

for split, df in dfs.items():
    print(f"{split}에 있는 질문 개수: {df['id'].nunique()}")

In [ ]:
dfs["train"].info()

레이블링된 데이터셋의 가치:
- 법률문서 데이터셋 CUAD는 13,000개 샘플을 레이블링할 법률전문가 고용비용을 고려할 때 약 2백만 달러의 가치가 있는 것으로 추정됨
- D. Hendrycks et al., “CUAD: An Expert-Annotated NLP Dataset for Legal Contract Review”, (2021).

<img alt="Column names" width="600" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_Table_1.png?raw=1" id="Table_1"/>

In [ ]:
qa_cols = ["title", "question", "answers.text",
           "answers.answer_start", "context"]
sample_df = dfs["train"][qa_cols].sample(2, random_state=7)
sample_df

- 질문이 문법적으로 바르지 않다
- 빈 answers.text 항목에는 리뷰에서 답을 찾지 못해 답변이 불가능한 질문이 담김
- 시작 인덱스와 answer span의 길이를 사용해 리뷰에서 답변에 해당하는 텍스트를 추출할 수 있다

In [ ]:
start_idx = sample_df["answers.answer_start"].iloc[0][0]
end_idx = start_idx + len(sample_df["answers.text"].iloc[0][0])
sample_df["context"].iloc[0][start_idx:end_idx]

훈련데이터셋에 대략 어떤 종류의 질문이 있는지 흔한 몇가지 단어로 시작하는 질문의 개수를 카운트해보자

In [ ]:
import matplotlib.pyplot as plt
counts = {}
question_types = ["What", "How", "Is", "Does", "Do", "Was", "Where", "Why"]

for q in question_types:
    counts[q] = dfs["train"]["question"].str.startswith(q).value_counts()[True]

pd.Series(counts).sort_values().plot.barh()
plt.title("Frequency of Question Types")
plt.show()

In [ ]:
for question_type in ["How", "What", "Is"]:
    for question in (
        dfs["train"][dfs["train"].question.str.startswith(question_type)]\
            .sample(n=3, random_state=42)['question']):
        print(question)

### 사이드바: The Stanford Question Answering Dataset SQuAD
- SubjQA의 (question, review, \[answer sentences\]) 형식: Exatractive QA에서 흔히 사용됨
- SQuAD에서 처음 사용된 형식
- P. Rajpurkar et al., “SQuAD: 100,000+ Questions for Machine Comprehension of Text”, (2016).
- 컴퓨터가 텍스트 문단을 읽고 관련 질문에 답변할 수 있는지를 테스트할 때 많이 사용됨
- SQuAD 1.1에 주어진 텍스트와 연관되지만 텍스트만으로는 대답할 수 없는 적대적인 질문(adversarial question)을 추가하여 SQuAD 2.0이 나옴
- P. Rajpurkar, R. Jia, and P. Liang, “Know What You Don’t Know: Unanswerable Questions for SQuAD”,
(2018).
- 2019년 이후 대부분의 모델이 사람의 능력(89.5 F1)을 뛰어 넘음
- 이것이 진정한 독해 능력을 나타내는 것은 아님.
- 구글 Natural Questions(NQ) 데이터셋(2019년): SQuAD보다 답변이 길고 더 어려움

<img alt="SQuAD SotA" width="600" caption="Progress on the SQuAD 2.0 benchmark (image from Papers with Code)" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_squad-sota.png?raw=1" id="squad-sota"/>

### 사이드바 끝

### 7.1.2 텍스트에서 답 추출하기
고객 리뷰에 있는 텍스트에서 답변에 사용할 만한 부분을 식별할 방법을 알려면 다음 방법을 이해해야 한다:
- 지도학습 문제로 구성하기
- QA 작업을 위해 텍스트를 토큰화하고 인코딩하기
- 모델의 최대 문맥 크기를 초과하는 긴 텍스트 다루기

#### 범위 분류(Span Classification)

<img alt="QA Head" width="600" caption="The span classification head for QA tasks" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_qa-head.png?raw=1" id="qa-head" style="margin-left: auto; margin-right: auto"/>
<p style="text-align: center;">Figure 7-4. QA 작업을 위한 Span Classification Head</p>

- 훈련셋에는 비교적 적은 1,295개의 샘플만 있으므로, SQuAD 같은 대규모 QA 데이터셋에서 Fine-tuning된 Language Model로 시작하는 것이 좋다.
- <span style="color:IndianRed">**[주목]** </span> 이것은 사전훈련된 모델로 task에 특화된 head를 fine-tuning하는 이전 장의 방식과 다르다.
- 클래스의 개수가 달라지면 분류 헤드를 미세조정해야 하지만, 추출적 QA는 레이블 구조가 데이터셋에따라 달라지지 않으므로 미세조정된 모델로 시작해도 된다.

<img alt="SQuAD models" width="600" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_Table_5.png?raw=1" id="squad-models"/>
<p style="text-align: center;">Figure 7-5. 허깅페이스에서 QA 모델 고르기</p>

#### MiniLM을 사용하기로 한다.
- 99%의 성능을 유지하면서 두 배 빠른 BERT 기반 압축 버전

#### QA를 위한 텍스트 토큰화

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "deepset/minilm-uncased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
question = "How much music can this hold?"
context = """An MP3 is about 1 MB/minute, so about 6000 hours depending on \
file size."""
inputs = tokenizer(question, context, return_tensors="pt")

In [ ]:
inputs.keys()

- PyTorch tensor가 반환된다. 이것을 forward pass시킬 것이다.
- 토큰화된 입력을 보면,

In [ ]:
input_df = pd.DataFrame.from_dict(tokenizer(question, context), orient="index")
input_df

`token_type_ids`: 0 은 질문 토큰, 1 은 문맥 토큰

In [ ]:
print(tokenizer.decode(inputs["input_ids"][0]))

이와 같이 QA 샘플마다 다음과 같은 format으로 입력이 구성된다.
> [CLS] question tokens [SEP] context tokens [SEP]  

첫번째 [SEP]의 위치는 token_type_ids에 의해 결정됨

QA Head와 함께 모델 객체를 초기화하고 입력을 forward pass 시킨다.

In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(model_ckpt)

with torch.no_grad():
    outputs = model(**inputs)
print(outputs)

In [ ]:
start_logits = outputs.start_logits
end_logits = outputs.end_logits

In [ ]:
print(f"입력 ID 크기: {inputs.input_ids.size()}")
print(f"시작 로짓 크기: {start_logits.size()}")
print(f"종료 로짓 크기: {end_logits.size()}")

In [ ]:
# 시작 토큰과 종료 토큰에 대한 예측 로짓. 오렌지 색 토큰이 가장 높은 점수를 가진 토큰입니다.
# 이 그래프는 다음을 참고했습니다. https://mccormickml.com/2020/03/10/question-answering-with-a-fine-tuned-BERT
import numpy as np
import matplotlib.pyplot as plt

s_scores = start_logits.detach().numpy().flatten()
e_scores = end_logits.detach().numpy().flatten()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

fig, (ax1, ax2) = plt.subplots(nrows=2, sharex=True)
colors = ["C0" if s != np.max(s_scores) else "C1" for s in s_scores]
ax1.bar(x=tokens, height=s_scores, color=colors)
ax1.set_ylabel("Start Scores")
colors = ["C0" if s != np.max(e_scores) else "C1" for s in e_scores]
ax2.bar(x=tokens, height=e_scores, color=colors)
ax2.set_ylabel("End Scores")
plt.xticks(rotation="vertical")
plt.show()

- **첫번째 그림**: 시작 토큰 후보 = 숫자 1과 6000 (질문이 숫자인 용량에 대한 질문이었으므로)
- **두번째 그림**: 종료 토큰 후보 = minute와 hours (시작 토큰과 연관된 토큰)

In [ ]:
import torch

start_idx = torch.argmax(start_logits)
end_idx = torch.argmax(end_logits) + 1
answer_span = inputs["input_ids"][0][start_idx:end_idx]
answer = tokenizer.decode(answer_span)
print(f"질문: {question}")
print(f"답변: {answer}")

토크나이저와 미세조정된 모델을 전달해 🤗Transformers 파이프라인을 초기화한다.

In [ ]:
from transformers import pipeline

pipe = pipeline("question-answering", model=model, tokenizer=tokenizer)
pipe(question=question, context=context, topk=3)

SubjQA에서 답변이 불가한 질문이면 answers.answer_start가 비어 있는 경우, [CLS] 토큰에 높은 시작 점수와 종료 점수를 할당하고, 파이프라인은 이 출력을 빈 문자열로 매핑한다.

In [ ]:
pipe(question="Why is there no data?", context=context,
     handle_impossible_answer=True)

- 이 예에서는 로짓에 argmax를 적용해 시작과 종료 인덱스를 구했다.
- 이 방법은 문맥 대신 질문에 속한 토큰을 선택하여 범위 밖으로 벗어난 답을 생성할 수 있다.
- 실전에서는, 범위 내 있는지, 시작 인덱스가 종료 인덱스 앞에 있는지 등의 다양한 제약 조건에 따라 파이프라인이 최상의 시작 인덱스와 종료 인덱스의 조합을 계산한다.

#### 긴 텍스트 다루기

- 다음 그림을 보면 SubjQA 훈련데이터셋의 상당 부분이 MiniLM의 문맥 크기인 512 토큰에 맞지 않는 질문-문맥 쌍을 갖는다.
- Sliding window를 적용하는 방법이 유용하다.

In [ ]:
# SubjQA 훈련 세트에 있는 질문-문맥 쌍의 토큰 분포
def compute_input_length(row):
    inputs = tokenizer(row["question"], row["context"])
    return len(inputs["input_ids"])

dfs["train"]["n_tokens"] = dfs["train"].apply(compute_input_length, axis=1)

fig, ax = plt.subplots()
dfs["train"]["n_tokens"].hist(bins=100, grid=False, ec="C0", ax=ax)
plt.xlabel("Number of tokens in question-context pair")
ax.axvline(x=512, ymin=0, ymax=1, linestyle="--", color="C1",
           label="Maximum sequence length")
plt.legend()
plt.ylabel("Count")
plt.show()

<img alt="Sliding window" width="600" caption="How the sliding window creates multiple question-context pairs for long documents—the first bar corresponds to the question, while the second bar is the context captured in each window" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_sliding-window.png?raw=1" id="sliding-window"/>
<p style="text-align: center;">Figure 7-8. 긴 문서에서 sliding window가 여러 개의 질문-문맥 쌍을 만드는 방법. 첫 번째(파란색) 막대는 질문에 해당하고, 두 번째(붉은색) 막대는 각 window에서 캡처한 문맥이다.</p>

🤗Transformers는 토크나이저에 `return_overflowing_tokens=True`를 설정해 슬라이딩 윈도를 만든다.
- max_seq_length: 윈도우 크기
- doc_stride: stride 크기

In [ ]:
example = dfs["train"].iloc[0][["question", "context"]]
tokenized_example = tokenizer(example["question"], example["context"],
                              return_overflowing_tokens=True, max_length=100,
                              stride=25)

In [ ]:
for idx, window in enumerate(tokenized_example["input_ids"]):
    print(f"#{idx} 윈도에는 {len(window)}개의 토큰이 있습니다.")

input_ids를 디코딩해서 두 윈도우가 어디서 겹치는지 확인한다

In [ ]:
for window in tokenized_example["input_ids"]:
    print(f"{tokenizer.decode(window)} \n")

End-to_end QA pipeline을 만드는데 필요한 다른 구성요소를 알아보자

### 7.1.3 Haystack을 사용해 QA Pipeline 구축하기
- Haystack: 대규모 언어 모델(LLM)을 사용하여 맞춤형 앱을 구축하기 위한 deepset의 오픈 소스 Python 프레임워크(https://haystack.deepset.ai/overview/intro)
> Haystack은 LLM(GPT-4, Falcon 등) 및 Transformer 모델을 사용하는 최첨단 NLP 시스템을 개발하기 위한 포괄적인 도구를 제공합니다. Haystack을 사용하면 Hugging Face, OpenAI, Cohere와 같은 플랫폼에서 호스팅되는 다양한 모델은 물론 SageMaker 및 로컬 모델에 배포된 모델까지 손쉽게 실험하여 사용 사례에 꼭 맞는 모델을 찾을 수 있습니다.\<from Haystack homepage\>

- 앞에서는 질문과 문맥을 모두 모델에 제공했지만, 실제 사용자는 제품에 대한 질문만 제공한다.
- 따라서, 말뭉치의 전체 리뷰 중에서 관련된 텍스트를 선택할 방법이 필요하다.
- 최신 QA 시스템은 Retriever-Reader 아키텍처를 기반으로 이를 처리한다.
  - <u>Retriever</u>:
    - __Sparse Retriever__: 단어 빈도를 사용해 각 문서와 query를 희소 벡터로 변환하고 이 벡터의 내적을 계산해 query와 문서의 관련성을 결정.
    - __Dense Retriever__: 트랜스포머 같은 인코더를 사용해 query와 문서를 contexualized embedding으로 표현. 임베딩에는 의미가 인코딩되므로 검색 정확도가 향상된다.
    
  - <u>Reader</u>:
    - Retriever가 제공한 문서에서 답을 추출
    - 독해 모델 또는 생성 모델

<img alt="QA Architecture" width="600" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_retriever-reader.png?raw=1" id="retriever-reader"/>
<p style="text-align: center;">Figure 7-9. 최신 QA 시스템의 Retriever-Reader 아키텍처</p>

Haystack을 사용한 QA 파이프라인의 추가 구성 요소
- <u>Document store</u>: 문서와 메타데이터를 저장하는 문서 전용 DB
- <u>Pipeline</u>: QA 시스템의 구성요소를 결합하고 여러 retriever에서 추출한 문서를 합치는 기능

#### 문서 저장소 초기화하기
- Haystack 저장소마다 조합할 수 있는 전용 retriever가 있다
- 이 장에서 희소 및 밀집 retriever 모두를 살펴보므로, 두 retriever type에 모두 호환되는 `ElasticsearchDocumentStore`를 사용한다.
- Elasticsearch: 텍스트, 수치, 지리 데이터, 구조적 데이터, 비구조적 데이터를 포함해 다양한 데이터 타입을 처리하는 검색 엔진. 업계 표준적.

<img alt="Haystack store" width="400" src="https://github.com/kikim6114/nlp2026-transformers/blob/main/images/chapter07_Table_3.png?raw=1" id="Haystack store"/>
<p style="text-align: center;">Table 7-3. Haystack 최신 QA 시스템의 Retriever-Reader 아키텍처</p>

# 강병민 수정
## Elasticsearch 서버 실행

다음 코드는 Python에서 Elasticsearch 서버를 실행하는 환경 설정 및 구동 과정이다.  
구체적인 실행 과정은 다음과 같다.

1. 사용자 권한 설정
    - 보안상의 이유로 Elasticsearch를 일반 사용자 계정(`es`)으로 실행하도록 별도의 계정을 생성하고, Elasticsearch 설치 디렉터리(`elasticsearch-7.9.2`)의 소유권을 이 계정으로 설정한다.

2. Elasticsearch 서버 실행
    - `subprocess.Popen`을 사용하여 일반 사용자(`es`) 권한으로 Elasticsearch 프로세스를 실행한다.
    - Elasticsearch 서버는 싱글 노드 모드(`discovery.type=single-node`)로 설정하여 로컬에서 간단한 검색 환경을 구축한다.

3. 서버 기동 대기
    - Elasticsearch가 정상적으로 초기화 및 기동될 수 있도록 약 30초간 프로세스 대기 시간을 설정한다.

4. 서버 연결 확인
    - 로컬 서버(`http://localhost:9200`)에 HTTP 요청을 보내어 Elasticsearch 서버의 기동 상태를 점검한다.
    - 정상 연결이 이루어지지 않은 경우, 프로세스 로그 및 Elasticsearch 로그 파일을 읽어 초기화 실패의 원인을 확인하도록 구성되어 있다.

In [ ]:
# 1) 'es' 라는 일반 계정 생성하고, ES 디렉토리 소유권을 es로 변경
!useradd -m es
!chown -R es:es elasticsearch-7.9.2
# 1) ES 로그 디렉터리 생성(혹시 없으면) 및 권한 완전 개방
!mkdir -p elasticsearch-7.9.2/logs
!chmod -R 777 elasticsearch-7.9.2/logs

In [ ]:
# url = """https://artifacts.elastic.co/downloads/elasticsearch/\
# elasticsearch-7.9.2-linux-x86_64.tar.gz"""
# !wget -nc -q {url}
# !tar -xzf elasticsearch-7.9.2-linux-x86_64.tar.gz

# Elasticsearch 다운로드 및 압축해제
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.9.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-7.9.2-linux-x86_64.tar.gz
!chmod +x elasticsearch-7.9.2/bin/elasticsearch

In [ ]:
# import os
# from subprocess import Popen, PIPE, STDOUT

# # 백그라운드 프로세스로 일래스틱서치를 실행합니다
# !chown -R daemon:daemon elasticsearch-7.9.2  # 백그라운드로 실행
# es_server = Popen(args=['elasticsearch-7.9.2/bin/elasticsearch'],
#                   stdout=PIPE, stderr=STDOUT, preexec_fn=lambda: os.setuid(1))
# # 일래스틱서치가 시작할 때까지 기다립니다
# !sleep 30

In [ ]:
# # 또는 도커가 설치되어 있다면
# from haystack.utils import launch_es

# launch_es()

연결 테스트

In [ ]:
# !curl -X GET "localhost:9200/?pretty"

In [ ]:
import subprocess, time, requests, os

# es 계정 생성 및 소유권 변경 (한 번만 해주세요)
!useradd -m es
!chown -R es:es elasticsearch-7.9.2

# Elasticsearch를 일반 사용자(es) 권한으로 실행
es = subprocess.Popen(
    [
      "runuser", "-u", "es", "--",
      "elasticsearch-7.9.2/bin/elasticsearch",
      "-E", "discovery.type=single-node"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Elasticsearch를 일반 사용자(es) 권한으로 실행 중입니다… 약 30초 기다려주세요.")
time.sleep(30)

# 연결 테스트
try:
    r = requests.get("http://localhost:9200")
    print(f"✅ 연결 성공: {r.json()}")
except Exception as e:
    # f-string 하나로만 인자를 전달
    print(f"❌ 연결 실패: {e}")

    # 프로세스가 죽었다면 stdout/stderr 로그 확인
    if es.poll() is not None:
        out, _ = es.communicate(timeout=5)
        print(f"\n--- ES stdout/stderr ---\n{out}")

    # elasticsearch.log 파일 직접 읽기

Elasticsearch 서버를 설치하고 실행했으므로, 문서 저장소 개체를 초기화한다.

In [ ]:
# document_store --> document_stores
from haystack.document_stores.elasticsearch import ElasticsearchDocumentStore

# 밀집 리트리버에서 사용할 문서 임베딩을 반환합니다.
document_store = ElasticsearchDocumentStore(return_embedding=True)

In [ ]:
# 노트북을 다시 시작할 때 일래스틱서치 저장소를 모두 비우는 것이 좋습니다.
if len(document_store.get_all_documents()) or len(document_store.get_all_labels()) > 0:
    document_store.delete_documents("document")
    document_store.delete_documents("label")

In [ ]:
for split, df in dfs.items():
    # 중복 리뷰를 제외시킵니다
    docs = [{"content": row["context"], "id": row["review_id"],
             "meta":{"item_id": row["title"], "question_id": row["id"],
                     "split": split}}
        for _,row in df.drop_duplicates(subset="context").iterrows()]
    document_store.write_documents(documents=docs, index="document")

print(f"{document_store.get_document_count()}개 문서가 저장되었습니다")

#### 리트리버 초기화하기

`BM25Retriever`: 고전적인 TF-IDF 알고리즘을 개선한 버전


헤이스택 1.4에서 `ElasticsearchRetriever`가 `BM25Retriever`로 바뀌었습니다. 여전히 버전 1.5에서 `ElasticsearchRetriever`를 사용할 수 있지만 향후 버전에서 삭제될 수 있습니다.

In [ ]:
from haystack.nodes.retriever import BM25Retriever

bm25_retriever = BM25Retriever(document_store=document_store)

In [ ]:
item_id = "B0074BW614"
query = "Is it good for reading?"
retrieved_docs = bm25_retriever.retrieve(
    query=query, top_k=3, filters={"item_id":[item_id], "split":["train"]})

In [ ]:
print(retrieved_docs[0])

#### 리더 초기화하기

In [ ]:
# from haystack.reader.farm import FARMReader
from haystack.nodes import FARMReader

# 백그라운드 프로세스로 일래스틱서치를 실행합니다
# !chown -R daemon:daemon elasticsearch-7.9.2  # 백그라운드로 실행
# es_server = Popen(args=['elasticsearch-7.9.2/bin/elasticsearch'],
#                   stdout=PIPE, stderr=STDOUT, preexec_fn=lambda: os.setuid(1))
# 일래스틱서치가 시작할 때까지 기다립니다
# !sleep 30
model_ckpt = "deepset/minilm-uncased-squad2"
max_seq_length, doc_stride = 384, 128
reader = FARMReader(model_name_or_path=model_ckpt, progress_bar=False,
                    max_seq_len=max_seq_length, doc_stride=doc_stride,
                    return_no_answer=True)

In [ ]:
print(reader.predict_on_texts(question=question, texts=[context], top_k=1))

#### 모두 합치기

In [ ]:
# from haystack.pipeline import ExtractiveQAPipeline
from haystack.pipelines import ExtractiveQAPipeline

pipe = ExtractiveQAPipeline(reader=reader, retriever=bm25_retriever)
print("✅ ExtractiveQAPipeline 초기화 성공")

In [ ]:
n_answers = 3
preds = pipe.run(query=query,
                 params={"Retriever": {"top_k": 3, "filters": {"item_id": [item_id], "split": ["train"]}},
                                      "Reader": {"top_k": n_answers}})

print(f"질문: {preds['query']} \n")
for idx in range(n_answers):
    print(f"답변 {idx+1}: {preds['answers'][idx].answer}")
    print(f"해당 리뷰 텍스트: ...{preds['answers'][idx].context}...")
    print("\n\n")

## QA 파이프라인 개선하기

### 리트리버 평가하기

In [ ]:
from haystack.pipelines import DocumentSearchPipeline

pipe = DocumentSearchPipeline(retriever=bm25_retriever)

In [ ]:
from haystack import Label, Answer, Document

labels = []
for i, row in dfs["test"].iterrows():
    # 리트리버에서 필터링을 위해 사용하는 메타데이터
    meta = {"item_id": row["title"], "question_id": row["id"]}
    # 답이 있는 질문을 레이블에 추가합니다
    if len(row["answers.text"]):
        for answer in row["answers.text"]:
            label = Label(
                query=row["question"], answer=Answer(answer=answer), origin="gold-label", document=Document(content=row["context"], id=row["review_id"]),
                meta=meta, is_correct_answer=True, is_correct_document=True,
                no_answer=False, filters={"item_id": [meta["item_id"]], "split":["test"]})
            labels.append(label)
    # 답이 없는 질문을 레이블에 추가합니다
    else:
        label = Label(
            query=row["question"], answer=Answer(answer=""), origin="gold-label", document=Document(content=row["context"], id=row["review_id"]),
            meta=meta, is_correct_answer=True, is_correct_document=True,
            no_answer=True, filters={"item_id": [row["title"]], "split":["test"]})
        labels.append(label)

In [ ]:
print(labels[0])

In [ ]:
document_store.write_labels(labels, index="label")
print(f"""{document_store.get_label_count(index="label")}개의 \
질문 답변 쌍을 로드했습니다.""")

In [ ]:
labels_agg = document_store.get_all_labels_aggregated(
    index="label",
    open_domain=True,
    aggregate_by_meta=["item_id"]
)
print(len(labels_agg))

In [ ]:
# 다음처럼 원하는 top_k 값으로 파이프라인을 실행할 수 있습니다
eval_result = pipe.eval(
    labels=labels_agg,
    params={"Retriever": {"top_k": 3}},
)
metrics = eval_result.calculate_metrics()

In [ ]:
print(f"재현율@3: {metrics['Retriever']['recall_single_hit']:.2f}")

In [ ]:
eval_df = eval_result["Retriever"]
eval_df[eval_df["query"] == "How do you like the lens?"][["query", "filters", "rank", "document_id", "gold_document_ids", "gold_id_match"]]

In [ ]:
def evaluate_retriever(retriever, topk_values = [1,3,5,10,20]):
    topk_results = {}
    # 최대 top_k를 계산합니다
    max_top_k = max(topk_values)
    # 파이프라인을 만듭니다
    p = DocumentSearchPipeline(retriever=retriever)
    # 테스트 세트에 있는 질문-답변 쌍을 순회하면서 최대 top_k로 추론을 실행합니다
    eval_result = p.eval(
        labels=labels_agg,
        params={"Retriever": {"top_k": max_top_k}},
    )
    # 각 top_k 값에 대해 재현율을 계산합니다
    for topk in topk_values:
        metrics = eval_result.calculate_metrics(simulated_top_k_retriever=topk)
        topk_results[topk] = {"recall": metrics["Retriever"]["recall_single_hit"]}

    return pd.DataFrame.from_dict(topk_results, orient="index")


bm25_topk_df = evaluate_retriever(bm25_retriever)

In [ ]:
def plot_retriever_eval(dfs, retriever_names):
    fig, ax = plt.subplots()
    for df, retriever_name in zip(dfs, retriever_names):
        df.plot(y="recall", ax=ax, label=retriever_name)
    plt.xticks(df.index)
    plt.ylabel("Top-k Recall")
    plt.xlabel("k")
    plt.show()

plot_retriever_eval([bm25_topk_df], ["BM25"])

#### DPR

<img alt="DPR Architecture" caption="DPR's bi-encoder architecture for computing the relevance of a document and query" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter07_dpr.png?raw=1" id="dpr"/>

# 강병민
# CustomDensePassageRetriever 클래스 정의 이유

기존의 `DensePassageRetriever` 클래스는 Haystack 내부에서 임베딩을 생성하는 과정에서 반복적으로 오류가 발생하는 문제가 있었다. 특히 Haystack의 내장 DPR(Dense Passage Retriever)의 `embed_queries()` 메서드가 내부적으로 임베딩을 제대로 생성하지 못하거나 빈 값을 반환하여 파이프라인 실행 중 다음과 같은 오류를 일으켰다.

Exception while running node ‘Retriever’: object of type ‘NoneType’ has no len()

본 문제를 해결하기 위해 다음과 같은 방식으로 `CustomDensePassageRetriever` 클래스를 새롭게 정의하였다.

- 부모 클래스(`DensePassageRetriever`)의 생성자를 명시적으로 호출하여 내부 속성 초기화를 올바르게 수행하였다.
- 임베딩 생성을 외부 Hugging Face DPR 모델로 직접 명시적으로 수행하여 Haystack 내부의 문제를 근본적으로 우회하였다.
- 성능과 메모리 효율성을 고려하여 `tokenizer` 및 `encoder` 객체를 클래스 초기화 시점에 단 한 번만 로드하고 재사용하도록 구성하였다.

이를 통해 임베딩 생성 과정의 안정성과 성능을 동시에 확보하였다.

```python
import torch
from haystack.nodes import DensePassageRetriever
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer

class CustomDensePassageRetriever(DensePassageRetriever):
    def __init__(self, document_store, query_embedding_model, passage_embedding_model, embed_title=False):
        super().__init__(
            document_store=document_store,
            query_embedding_model=query_embedding_model,
            passage_embedding_model=passage_embedding_model,
            embed_title=embed_title
        )
        
        # tokenizer와 encoder는 한 번만 초기화
        self.tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(query_embedding_model)
        self.encoder = DPRQuestionEncoder.from_pretrained(query_embedding_model).to("cuda").eval()

    def embed_queries(self, queries: list):
        embeddings = []
        with torch.no_grad():
            for q in queries:
                inputs = self.tokenizer(q, return_tensors="pt", truncation=True, max_length=64).to("cuda")
                emb = self.encoder(**inputs).pooler_output[0].cpu().numpy()
                embeddings.append(emb)
        return embeddings

위와 같은 방법으로 정의된 CustomDensePassageRetriever 클래스는 기존 클래스에서 발생하던 내부 임베딩 처리 오류를 효과적으로 해결하였다.

In [ ]:
'''
# 변경 전
from haystack.retriever.dense import DensePassageRetriever
dpr_retriever = DensePassageRetriever(
    document_store=document_store,
    query_embedding_model="facebook/dpr-question_encoder-single-nq-base",
    passage_embedding_model="facebook/dpr-ctx_encoder-single-nq-base",
    embed_title=False)


document_store.update_embeddings(retriever=dpr_retriever)
'''
# 변경 후
# 클래스 직접 지정

import torch
from haystack.nodes import DensePassageRetriever
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer

class CustomDensePassageRetriever(DensePassageRetriever):
    def __init__(self, document_store, query_embedding_model, passage_embedding_model, embed_title=False):
        super().__init__(
            document_store=document_store,
            query_embedding_model=query_embedding_model,
            passage_embedding_model=passage_embedding_model,
            embed_title=embed_title
        )

        # tokenizer와 encoder는 한 번만 초기화
        self.tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(query_embedding_model)
        self.encoder = DPRQuestionEncoder.from_pretrained(query_embedding_model).to("cuda").eval()

    def embed_queries(self, queries: list):
        embeddings = []
        with torch.no_grad():
            for q in queries:
                inputs = self.tokenizer(q, return_tensors="pt", truncation=True, max_length=64).to("cuda")
                emb = self.encoder(**inputs).pooler_output[0].cpu().numpy()
                embeddings.append(emb)
        return embeddings


dpr_retriever = CustomDensePassageRetriever(
    document_store=document_store,
    query_embedding_model="facebook/dpr-question_encoder-single-nq-base",
    passage_embedding_model="facebook/dpr-ctx_encoder-single-nq-base",
    embed_title=False
)


document_store.update_embeddings(retriever=dpr_retriever)

In [ ]:
dpr_topk_df = evaluate_retriever(dpr_retriever)
plot_retriever_eval([bm25_topk_df, dpr_topk_df], ["BM25", "DPR"])

### 리더 평가하기

In [ ]:
from haystack.modeling.evaluation.squad import compute_f1, compute_exact

pred = "about 6000 hours"
label = "6000 hours"
print(f"EM: {compute_exact(label, pred)}")
print(f"F1: {compute_f1(label, pred)}")

In [ ]:
pred = "about 6000 dollars"
print(f"EM: {compute_exact(label, pred)}")
print(f"F1: {compute_f1(label, pred)}")

In [ ]:
from haystack.pipelines import Pipeline
def evaluate_reader(reader):
    score_keys = ['exact_match', 'f1']
    p = Pipeline()
    p.add_node(component=reader, name="Reader", inputs=["Query"])

    eval_result = p.eval(
        labels=labels_agg,
        documents= [[label.document for label in multilabel.labels] for multilabel in labels_agg],
        params={},
    )
    metrics = eval_result.calculate_metrics(simulated_top_k_reader=1)

    return {k:v for k,v in metrics["Reader"].items() if k in score_keys}

reader_eval = {}
reader_eval["Fine-tune on SQuAD"] = evaluate_reader(reader)

In [ ]:
def plot_reader_eval(reader_eval):
    fig, ax = plt.subplots()
    df = pd.DataFrame.from_dict(reader_eval)
    df.plot(kind="bar", ylabel="Score", rot=0, ax=ax)
    ax.set_xticklabels(["EM", "F1"])
    plt.legend(loc='upper left')
    plt.show()

plot_reader_eval(reader_eval)

### 도메인 적응

<img alt="SQuAD Schema" caption="Visualization of the SQuAD JSON format" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter07_squad-schema.png?raw=1" id="squad-schema"/>

In [ ]:
def create_paragraphs(df):
    paragraphs = []
    id2context = dict(zip(df["review_id"], df["context"]))
    for review_id, review in id2context.items():
        qas = []
        # 특정 문맥으로 전체 질문-답 쌍을 필터링합니다
        review_df = df.query(f"review_id == '{review_id}'")
        id2question = dict(zip(review_df["id"], review_df["question"]))
        # qas 배열을 만듭니다
        for qid, question in id2question.items():
            # 하나의 질문 ID에 대해 필터링합니다
            question_df = df.query(f"id == '{qid}'").to_dict(orient="list")
            ans_start_idxs = question_df["answers.answer_start"][0].tolist()
            ans_text = question_df["answers.text"][0].tolist()
            # 답변 가능한 질문을 추가합니다
            if len(ans_start_idxs):
                answers = [
                    {"text": text, "answer_start": answer_start}
                    for text, answer_start in zip(ans_text, ans_start_idxs)]
                is_impossible = False
            else:
                answers = []
                is_impossible = True
            # 질문-답 쌍을 qas에 추가합니다
            qas.append({"question": question, "id": qid,
                        "is_impossible": is_impossible, "answers": answers})
        # 문맥과 질문-답 쌍을 paragraphs에 추가합니다
        paragraphs.append({"qas": qas, "context": review})
    return paragraphs

In [ ]:
product = dfs["train"].query("title == 'B00001P4ZH'")
create_paragraphs(product)

```python
[{'qas': [{'question': 'How is the bass?',
    'id': '2543d296da9766d8d17d040ecc781699',
    'is_impossible': True,
    'answers': []}],
  'context': 'I have had Koss headphones ...',
    'id': 'd476830bf9282e2b9033e2bb44bbb995',
    'is_impossible': False,
    'answers': [{'text': 'Bass is weak as expected', 'answer_start': 1302},
     {'text': 'Bass is weak as expected, even with EQ adjusted up',
      'answer_start': 1302}]}],
  'context': 'To anyone who hasn\'t tried all ...'},
 {'qas': [{'question': 'How is the bass?',
    'id': '455575557886d6dfeea5aa19577e5de4',
    'is_impossible': False,
    'answers': [{'text': 'The only fault in the sound is the bass',
      'answer_start': 650}]}],
  'context': "I have had many sub-$100 headphones ..."}]
```

In [ ]:
import json

def convert_to_squad(dfs):
    for split, df in dfs.items():
        subjqa_data = {}
        # 각 제품 ID에 대해 `paragraphs`를 만듭니다
        groups = (df.groupby("title").apply(create_paragraphs)
            .to_frame(name="paragraphs").reset_index())
        subjqa_data["data"] = groups.to_dict(orient="records")
        # 결과를 디스크에 저장합니다
        with open(f"electronics-{split}.json", "w+", encoding="utf-8") as f:
            json.dump(subjqa_data, f)

convert_to_squad(dfs)

In [ ]:
train_filename = "electronics-train.json"
dev_filename = "electronics-validation.json"

reader.train(data_dir=".", use_gpu=True, n_epochs=1, batch_size=16,
             train_filename=train_filename, dev_filename=dev_filename)

In [ ]:
reader_eval["Fine-tune on SQuAD + SubjQA"] = evaluate_reader(reader)
plot_reader_eval(reader_eval)

In [ ]:
minilm_ckpt = "microsoft/MiniLM-L12-H384-uncased"
minilm_reader = FARMReader(model_name_or_path=minilm_ckpt, progress_bar=False,
                           max_seq_len=max_seq_length, doc_stride=doc_stride,
                           return_no_answer=True)

In [ ]:
minilm_reader.train(data_dir=".", use_gpu=True, n_epochs=1, batch_size=16,
             train_filename=train_filename, dev_filename=dev_filename)

In [ ]:
reader_eval["Fine-tune on SubjQA"] = evaluate_reader(minilm_reader)
plot_reader_eval(reader_eval)

### 전체 QA 파이프라인 평가하기

In [ ]:
from haystack.pipelines import ExtractiveQAPipeline
pipe = ExtractiveQAPipeline(retriever=bm25_retriever, reader=reader)

# 평가하기!
eval_result = pipe.eval(
    labels=labels_agg,
    params={},
)
metrics = eval_result.calculate_metrics(simulated_top_k_reader=1)
# 리더에서 지표를 추출합니다
reader_eval["QA Pipeline (top-1)"] = {
    k:v for k,v in metrics["Reader"].items()
    if k in ["exact_match", "f1"]}

In [ ]:
# 리더와 전체 QA 파이프라인의 EM과 F1-점수 비교
plot_reader_eval({"Reader": reader_eval["Fine-tune on SQuAD + SubjQA"],
                  "QA pipeline (top-1)": reader_eval["QA Pipeline (top-1)"]})

In [ ]:
# 또는 QA 파이프라인과 리더 지표를 한번에 얻습니다
# 리더 평가는 시뮬레이트된 완벽한 리트리버 결과를 사용해 두 번째로 실행됩니다
eval_result = pipe.eval(
    labels=labels_agg,
    params={},
    add_isolated_node_eval=True
)
metrics = eval_result.calculate_metrics(simulated_top_k_reader=1)
# 시뮬레이트된 완벽한 리트리버로 격리되어 실행된 리더로부터 지표를 추출합니다
isolated_metrics = eval_result.calculate_metrics(simulated_top_k_reader=1, eval_mode="isolated")

pipeline_reader_eval = {}
pipeline_reader_eval["Reader"] = {
    k:v for k,v in isolated_metrics["Reader"].items()
    if k in ["exact_match", "f1"]}
pipeline_reader_eval["QA Pipeline (top-1)"] = {
    k:v for k,v in metrics["Reader"].items()
    if k in ["exact_match", "f1"]}

plot_reader_eval(pipeline_reader_eval)

## 추출적 QA를 넘어서

<img alt="RAG Architecture" width="600" caption="The RAG architecture for fine-tuning a retriever and generator end-to-end (courtesy of Ethan Perez)" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter07_rag-architecture.png?raw=1" id="rag"/>

In [ ]:
import importlib.metadata
print(importlib.metadata.version("farm-haystack"))

In [ ]:
'''
원본코드
from haystack.generator.transformers import RAGenerator

generator = RAGenerator(model_name_or_path="facebook/rag-token-nq",
                        embed_title=False, num_beams=5)

Haystack 1.x 계열(최종 버전 1.26.4)에서는 RAGenerator 컴포넌트가 이미 제거된 상태이기 때문에,
더 이상 직접 import 할 수 없습니다.
대신 PromptNode사용
'''

from haystack.nodes import PromptNode, PromptTemplate
from haystack.pipelines import Pipeline


prompt_node = PromptNode(model_name_or_path="google/flan-t5-base")

In [ ]:
'''
원본코드
from haystack.pipeline import GenerativeQAPipeline
pipe = GenerativeQAPipeline(generator=generator, retriever=dpr_retriever)
'''
# Prompt 템플릿 정의
template = PromptTemplate(
    prompt="""Given the context, answer the question.
Context: {join(documents)};
Question: {query};
Answer:"""
)

prompt_node.default_prompt_template = template

# 파이프라인 구성 (Retriever → PromptNode)
pipe = Pipeline()
pipe.add_node(component=dpr_retriever, name="Retriever", inputs=["Query"])
pipe.add_node(component=prompt_node, name="Generator", inputs=["Retriever"])

In [ ]:
'''
def generate_answers(query, top_k_generator=3):
    preds = pipe.run(query=query,
                     params={"Retriever": {"top_k":5,
                                  "filters":{"item_id": ["B0074BW614"]}},
                             "Generator": {"top_k": top_k_generator}})
    print(f"질문: {preds['query']} \n")
    for idx in range(top_k_generator):
        # print(f"답변 {idx+1}: {preds['answers'][idx]['answer']}")
        print(f"답변 {idx+1}: {preds['answers'][idx].answer}")
'''
def generate_answers(query, top_k_retriever=5):
    preds = pipe.run(
        query=query,
        params={"Retriever": {"top_k": top_k_retriever, "filters":{"item_id": ["B0074BW614"]}}}
    )

    print(f"질문: {query}\n")
    answer = preds["results"][0]
    print(f"답변: {answer}\n")

In [ ]:
query

In [ ]:
generate_answers(query)

In [ ]:
generate_answers("What is the main drawback?")

## 결론

<img alt="QA Pyramid" caption="The QA hierarchy of needs" src="https://github.com/rickiepark/nlp-with-transformers/blob/main/images/chapter07_qa-pyramid.png?raw=1" id="qa-pyramid"/>